# Pytorch 101

In [1]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

## Tensor Fundatmentals

Tensor is an n-dimensional array with:
- shape (ex: [B, C, H, W])
- dtype (ex: floate32, float16, int64)
- device (cpu, cuda:0)
- grad tracking (requires_grad=True)

In [ ]:
# various ways to create tensors

torch.tensor(
    data=[1.0, 2.0],
    dtype=torch.float32,
    device="cpu",
    requires_grad=True
)

tensors = {
    "x"     : torch.tensor([[1.0, 2.0], [3.0, 4.0]]),       # from lists
    "n"     : torch.from_numpy(np.array([[1, 2], [2, 3]])), # from numpy
    "z"     : torch.zeros(3, 4, dtype=torch.float32),       # filled with zeroes
    "o"     : torch.ones(2, 2),                             # filled with ones
    "r"     : torch.randn(5, 3),                            # filled randomly from  N(0, 1)
    "u"     : torch.rand(5, 3),                             # filled randomly from  U(0, 1)
    "a"     : torch.arange(0, 10, 2),                       # [0,2,4,6,8]
    "l"     : torch.linspace(0, 10, 5),                     # [0,2,4,6,8]
    "eye"   : torch.eye(4)                                  # identity matrix
}       

for key, tensor in tensors.items():
    print(f"{key}: ", tensor)

x:  tensor([[1., 2.],
        [3., 4.]])
n:  tensor([[1, 2],
        [2, 3]])
z:  tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
o:  tensor([[1., 1.],
        [1., 1.]])
r:  tensor([[ 1.0556,  0.3030,  2.0077],
        [ 0.0217,  0.1045,  0.2809],
        [ 0.6371,  2.1561, -0.0353],
        [-1.0405, -0.0290, -0.0589],
        [-1.9591, -0.0022, -0.6105]])
u:  tensor([[0.3462, 0.9110, 0.0777],
        [0.3859, 0.8657, 0.1376],
        [0.8932, 0.9132, 0.4764],
        [0.3650, 0.7787, 0.0668],
        [0.8401, 0.1034, 0.7757]])
a:  tensor([0, 2, 4, 6, 8])
l:  tensor([ 0.0000,  2.5000,  5.0000,  7.5000, 10.0000])
eye:  tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])


In [27]:
# tensor status

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ", device)
print("type: ", device.type)
print("index: ", device.index)

x = torch.randn(2, 3, device=device, dtype=torch.float32)
x = x.to(torch.float16)          # dtype cast
x = x.to("cpu")                  # move
print(x)

print(f"x: {x}")
print(f"is contiguous: {x.is_contiguous()}, \nstride: {x.stride()}")  
x = x.transpose(0, 1)
print(f"x: {x}")
print(f"is contiguous: {x.is_contiguous()}, \nstride: {x.stride()}")  
x = x.contiguous()
print(f"x: {x}")
print(f"is contiguous: {x.is_contiguous()}, \nstride: {x.stride()}")  

device:  cpu
type:  cpu
index:  None
tensor([[-0.1609, -1.2109, -0.1265],
        [ 0.1808, -0.3594, -1.2188]], dtype=torch.float16)
x: tensor([[-0.1609, -1.2109, -0.1265],
        [ 0.1808, -0.3594, -1.2188]], dtype=torch.float16)
is contiguous: True, 
stride: (3, 1)
x: tensor([[-0.1609,  0.1808],
        [-1.2109, -0.3594],
        [-0.1265, -1.2188]], dtype=torch.float16)
is contiguous: False, 
stride: (1, 3)
x: tensor([[-0.1609,  0.1808],
        [-1.2109, -0.3594],
        [-0.1265, -1.2188]], dtype=torch.float16)
is contiguous: True, 
stride: (2, 1)


In [ ]:
# tensor shape operations

x = torch.rand(2, 5)            # batch of vectors
y = torch.rand(3, 2, 5)         # batch of sequences
z = torch.rand(2, 3, 2, 5)      # batch of images
print(f"shape of x: {y.shape}")

reshaped_tensors = {
    "y_t" : y.transpose(1, 2),  # swap dims
    "y_p" : y.permute(2, 1, 0), # arbitrary reorder
    "y_r" : y.reshape(2, 15),   # shape change (may copy)
    "y_v" : y.view(2, 15),      # shape change (no copy if contiguous)
}

for key, tensor in reshaped_tensors.items():
    print(f"shape of {key}: {tensor.shape}")

shape of x: torch.Size([3, 2, 5])
shape of y_t: torch.Size([3, 5, 2])
shape of y_p: torch.Size([5, 2, 3])
shape of y_r: torch.Size([2, 15])
shape of y_v: torch.Size([2, 15])


In [5]:
# Broadcasting

x = torch.randn(32, 128)         # (B, D)
b = torch.randn(128)             # (D,)
y = x + b                        # b broadcasts to (B, D)
print(f"shape of b: {b.shape}")
print(f"shape of y: {y.shape}")

shape of b: torch.Size([128])
shape of y: torch.Size([32, 128])


In [6]:
# Indexing and Masking

x = torch.randn(2, 5)
print(f"x: {x}")

indexed_tensors = {
    "row0"      : x[0],
    "cols_1_3"  : x[:, 1:4],
    "masking"   : x[x[:, 0] > 0]
}

for key, tensor in indexed_tensors.items():
    print(f"{key}: {tensor}")

x: tensor([[ 0.3778,  0.4211, -1.2698,  0.5023,  1.4780],
        [-0.0797,  1.0331, -1.5301,  2.6242, -0.3255]])
row0: tensor([ 0.3778,  0.4211, -1.2698,  0.5023,  1.4780])
cols_1_3: tensor([[ 0.4211, -1.2698,  0.5023],
        [ 1.0331, -1.5301,  2.6242]])
masking: tensor([[ 0.3778,  0.4211, -1.2698,  0.5023,  1.4780]])


In [28]:
# Tensor Operations

A = torch.randn(64, 128)
B = torch.randn(128, 256)

C = A @ B                        # matrix multiplication
print(f"shape of C: {C.shape}")

v = torch.randn(128)
y = A @ v                        # matrix vector multiplication
print(f"shape of y: {y.shape}")

dot = torch.dot(v, v)            # dot (inner) product
print(dot)

shape of C: torch.Size([64, 256])
shape of y: torch.Size([64])
tensor(161.9104)


In [31]:
# torch.linalg Operations

def torch_linalg_norms():
    print("\n=== Norms ===")
    x = torch.tensor([3.0, 4.0])

    # Vector norms
    print("vector_norm L2:", torch.linalg.vector_norm(x, ord=2))  # 5
    print("vector_norm L1:", torch.linalg.vector_norm(x, ord=1))  # 7

    # torch.linalg.norm can also do vector norms (depending on dim usage)
    print("norm (vector):", torch.linalg.norm(x))  # defaults to 2-norm for vectors

    A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

    # Frobenius norm (common matrix norm)
    print("matrix_norm Fro:", torch.linalg.matrix_norm(A, ord="fro"))

    # Spectral norm (largest singular value)
    print("matrix_norm 2:", torch.linalg.matrix_norm(A, ord=2))

    # Other useful matrix norms:
    # ord=1 => max column sum; ord=inf => max row sum
    print("matrix_norm 1:", torch.linalg.matrix_norm(A, ord=1))
    print("matrix_norm inf:", torch.linalg.matrix_norm(A, ord=float("inf")))

def torch_linalg_solve_family():
    print("\n=== Solving Linear Systems ===")

    # Square solve: AX = B
    A = torch.tensor([[3.0, 1.0], [1.0, 2.0]])
    B = torch.tensor([[9.0], [8.0]])
    X = torch.linalg.solve(A, B)
    print("solve X:\n", X)
    print("check A@X:\n", A @ X)

    # Triangular solve: TX = B
    # Make an upper-triangular T
    T = torch.tensor([[2.0, 1.0, -1.0],
                      [0.0, 3.0,  2.0],
                      [0.0, 0.0,  4.0]])
    b = torch.tensor([1.0, 7.0, 8.0])

    x = torch.linalg.solve_triangular(T, b, upper=True)
    print("solve_triangular x:\n", x)
    print("check T@x:\n", T @ x)

    # Least squares: min ||AX - B||_2
    # Example: fit y ≈ m*x + c from noisy points
    x_data = torch.tensor([0.0, 1.0, 2.0, 3.0])
    y_data = torch.tensor([1.0, 3.1, 5.0, 7.05])

    # Design matrix [x, 1]
    A_ls = torch.stack([x_data, torch.ones_like(x_data)], dim=1)  # shape (N,2)
    B_ls = y_data.unsqueeze(1)  # shape (N,1)

    ls = torch.linalg.lstsq(A_ls, B_ls)
    theta = ls.solution  # [m, c]
    m, c = theta.squeeze().tolist()
    print(f"lstsq solution: m={m:.4f}, c={c:.4f}")
    print("residual norm:", torch.linalg.vector_norm(A_ls @ theta - B_ls).item())

def torch_linalg_svd():
    print("\n=== SVD ===")
    A = torch.tensor([[3.0, 1.0, 1.0],
                      [-1.0, 3.0, 1.0]])

    # Full SVD: A = U diag(S) Vh
    U, S, Vh = torch.linalg.svd(A, full_matrices=False)
    print("U:\n", U)
    print("S:\n", S)
    print("Vh:\n", Vh)

    # Reconstruct
    A_recon = U @ torch.diag(S) @ Vh
    print("reconstruction error:", torch.linalg.matrix_norm(A - A_recon).item())

    # Singular values only
    S_only = torch.linalg.svdvals(A)
    print("svdvals:\n", S_only)

    # Low-rank approx (rank-1)
    k = 1
    A_rank1 = U[:, :k] @ torch.diag(S[:k]) @ Vh[:k, :]
    print("rank-1 approx:\n", A_rank1)

def torch_linalg_inverse_and_pinv():
    print("\n=== Inverse & Pseudoinverse ===")

    A = torch.tensor([[4.0, 7.0],
                      [2.0, 6.0]])
    A_inv = torch.linalg.inv(A)
    print("inv(A):\n", A_inv)
    print("A @ inv(A):\n", A @ A_inv)

    # Prefer solve for AX=B
    B = torch.tensor([[1.0], [0.0]])
    X_solve = torch.linalg.solve(A, B)
    X_via_inv = A_inv @ B
    print("X via solve:\n", X_solve)
    print("X via inv @ B:\n", X_via_inv)

    # Pseudoinverse: works for non-square / rank-deficient
    M = torch.tensor([[1.0, 2.0, 3.0],
                      [2.0, 4.0, 6.0]])  # rank-deficient (row2=2*row1)
    M_pinv = torch.linalg.pinv(M)
    print("pinv(M) shape:", M_pinv.shape)  # (3,2)

    # Minimum-norm least squares solution to Mx ≈ b
    b = torch.tensor([1.0, 2.0])
    x = M_pinv @ b
    print("x from pinv:", x)
    print("M @ x:", M @ x)

def torch_linalg_decompositions():
    print("\n=== Decompositions ===")

    # Cholesky: SPD matrix A = L L^T (or L L^H)
    A_spd = torch.tensor([[4.0, 1.0, 1.0],
                          [1.0, 3.0, 0.0],
                          [1.0, 0.0, 2.0]])
    L = torch.linalg.cholesky(A_spd)
    print("Cholesky L:\n", L)
    print("reconstruction error:",
          torch.linalg.matrix_norm(A_spd - L @ L.T).item())

    # QR: A = Q R
    A = torch.randn(5, 3)
    Q, R = torch.linalg.qr(A, mode="reduced")
    print("Q shape:", Q.shape, "R shape:", R.shape)
    print("orthogonality error:", torch.linalg.matrix_norm(Q.T @ Q - torch.eye(3)).item())
    print("reconstruction error:", torch.linalg.matrix_norm(A - Q @ R).item())

    # Eigh: symmetric / Hermitian eigen-decomposition
    S = torch.tensor([[2.0, 1.0],
                      [1.0, 2.0]])  # symmetric
    evals, evecs = torch.linalg.eigh(S)
    print("eigh evals:", evals)
    print("eigh evecs:\n", evecs)
    # Reconstruct: S = V diag(evals) V^T
    S_recon = evecs @ torch.diag(evals) @ evecs.T
    print("eigh recon error:", torch.linalg.matrix_norm(S - S_recon).item())

    # Eig: general square matrix (can be complex)
    G = torch.tensor([[0.0, -1.0],
                      [1.0,  0.0]])  # rotation by 90 degrees; eigenvalues are +/- i
    evals_g, evecs_g = torch.linalg.eig(G)
    print("eig evals:", evals_g)  # complex
    # Reconstruct: G = V diag(evals) V^{-1}
    G_recon = evecs_g @ torch.diag(evals_g) @ torch.linalg.inv(evecs_g)
    print("eig recon error:", torch.linalg.matrix_norm(G.to(torch.complex64) - G_recon).item())

def torch_linalg_determinants():
    print("\n=== det & slogdet ===")
    A = torch.tensor([[3.0, 1.0],
                      [2.0, 4.0]])

    d = torch.linalg.det(A)
    print("det(A):", d.item())

    sign, logabsdet = torch.linalg.slogdet(A)
    print("slogdet sign:", sign.item(), "logabsdet:", logabsdet.item())

    # Recover determinant (may be more stable than directly calling det for large matrices)
    det_from_slogdet = sign * torch.exp(logabsdet)
    print("det from slogdet:", det_from_slogdet.item())

def torch_linalg_diagnostics():
    print("\n=== cond & matrix_rank ===")

    # Condition number: large => ill-conditioned
    A = torch.tensor([[1.0, 1.0],
                      [1.0, 1.000001]])
    kappa2 = torch.linalg.cond(A, p=2)
    print("cond_2(A):", kappa2.item())

    # Numerical matrix rank (based on singular values threshold)
    M = torch.tensor([[1.0, 2.0, 3.0],
                      [2.0, 4.0, 6.0],
                      [1.0, 1.0, 1.0]])
    r = torch.linalg.matrix_rank(M)
    print("matrix_rank(M):", int(r.item()))


torch.set_printoptions(precision=4, sci_mode=False)
# torch_linalg_norms()
# torch_linalg_solve_family()
# torch_linalg_svd()
# torch_linalg_inverse_and_pinv()
# torch_linalg_decompositions()
# torch_linalg_determinants()
# torch_linalg_diagnostics()

## Automatic Differentiation

PyTorch autograd is (mostly) a reverse-mode automatic differentiation system over a dynamic computation graph.  
If a tensor has requires_grad=True, operations on it are tracked so gradients can be computed via reverse-mode autodiff.  

Internally it does two big things:
- During forward, it records a graph of ops (nodes) and tensor dependencies (edges).
- During backward, it walks that graph in reverse and propagates “gradient signals” using the chain rule.

In [ ]:
# Gradient of Scalars

x = torch.randn(3, requires_grad=True)
print(f"x: {x}")

y = (x**2).sum()        # y = Σ x_i^2
print(f"y: {y}")

y.backward()            # x.grad = dy/dx = 2x
print(f"grad of x: {x.grad}")

x: tensor([0.4742, 0.2338, 0.1147], requires_grad=True)
y: 0.2926923632621765
grad of x: tensor([0.9484, 0.4676, 0.2293])


In [31]:
# Gradient of Non-Scalars
 
x = torch.randn(3, requires_grad=True)
print(f"x: {x}")

y = x**2                # y = [x1^2, x2^2, x3^2]
print(f"y: {y}")

g = torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y))[0]
print(f"g: {g}")
print(x.grad)

# y.backward(x)
# print(f"x: {x.grad}")

x: tensor([-1.5376, -0.2645, -1.3159], requires_grad=True)
y: tensor([2.3643, 0.0700, 1.7317], grad_fn=<PowBackward0>)
g: tensor([-3.0753, -0.5291, -2.6319])
None


## Examples

### MLP Classifier

In [ ]:
# 1. Initialzie MLP Classifier 

class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=32, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = MLP()
print(model)

MLP(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
)


In [ ]:
# 2. Generate Data

N = 2000
X = torch.randn(N, 2)
y = (X[:, 0] + X[:, 1] > 0).long()   # label 1 if x0+x1 positive else 0

ds = TensorDataset(X, y)
dl = DataLoader(ds, batch_size=64, shuffle=True)
print(ds.tensors, ds.__annotations__)
# print(ds.__annotations__.)

(tensor([[ 0.4319, -0.1489],
        [ 0.8654, -0.8371],
        [-1.3048,  1.2216],
        ...,
        [-0.1268,  0.4419],
        [-1.2047, -1.3735],
        [-0.6761,  0.2470]]), tensor([1, 1, 0,  ..., 1, 0, 0])) {'tensors': typing.Tuple[torch.Tensor, ...]}


In [ ]:
# 3. Loss + optimizer

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# 4. Train
model.train()
for epoch in range(20):
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in dl:
        logits = model(xb)                 # forward
        loss = criterion(logits, yb)       # compute loss

        optimizer.zero_grad()              # clear old grads
        loss.backward()                    # backward (compute grads)
        optimizer.step()                   # update weights

        total_loss += loss.item() * xb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)

    print(f"epoch={epoch} loss={total_loss/total:.4f} acc={correct/total:.3f}")

# 5. Inference (no grad)
model.eval()
with torch.no_grad():
    sample = torch.tensor([[0.2, 0.1], [-1.0, 0.2]])
    probs = torch.softmax(model(sample), dim=1)
    print("probs:\n", probs)

epoch=0 loss=0.0282 acc=0.993
epoch=1 loss=0.0253 acc=0.993
epoch=2 loss=0.0226 acc=0.994
epoch=3 loss=0.0197 acc=0.996
epoch=4 loss=0.0178 acc=0.995
epoch=5 loss=0.0181 acc=0.994
epoch=6 loss=0.0164 acc=0.996
epoch=7 loss=0.0158 acc=0.996
epoch=8 loss=0.0146 acc=0.996
epoch=9 loss=0.0155 acc=0.996
epoch=10 loss=0.0153 acc=0.996
epoch=11 loss=0.0166 acc=0.994
epoch=12 loss=0.0142 acc=0.997
epoch=13 loss=0.0116 acc=0.998
epoch=14 loss=0.0138 acc=0.996
epoch=15 loss=0.0184 acc=0.992
epoch=16 loss=0.0159 acc=0.993
epoch=17 loss=0.0129 acc=0.997
epoch=18 loss=0.0103 acc=0.998
epoch=19 loss=0.0125 acc=0.994
probs:
 tensor([[1.0087e-07, 1.0000e+00],
        [1.0000e+00, 2.1876e-10]])


In [ ]:
# 6. Save Model
torch.save({
    "model": model.state_dict(),
    "optim": optimizer.state_dict(),
}, "tiny_mlp.pt")

In [ ]:
model2 = MLP()
model2.load_state_dict(torch.load("mlp.pt", map_location="cpu"))
model2.eval()